# Cosmological zoom-in — Part 3: zoom with baryons

| Step | Code | What it does | Part |
|---|---|---|---|
| 1–3 | MUSIC2, RAMSES, hast | Parent box, halo selection | 1 |
| 4–6 | MUSIC2, RAMSES, hast | Dark-matter-only zoom, decontamination | 2 |
| 7 | MUSIC2 | Zoom ICs with baryons, on the decontaminated region | **3** |
| 8 | RAMSES | Hydrodynamical zoom run | **3** |

We zoom on the same halo, now with gas, cooling, star formation and supernova feedback. The region is `results_part_wide` from Part 2, and the resolution is the same (`levelmax = 9` in MUSIC). For the laptop, the run stops at $z = 2$.

## 7. Zoom initial conditions with baryons

### 7.1. MUSIC parameter file

The file is `MyFirstZoom/ZoomHydro/ICs/music.conf`. It is the dark-matter-only `music.conf` of Part 2 with these changes:

| Parameter | Part 2 | Part 3 | Why |
|---|---|---|---|
| `baryons` | `no` | `yes` | MUSIC also writes the gas density and velocity (`ic_deltab`, `ic_velbx/y/z`), and the zoom tracer `ic_pvar_*` (see 7.2). |
| `region_point_file` | `../../Selection/results/music_zoom_2` | `../../ZoomDM/Decontaminate/results_part_wide` | The decontaminated region of Part 2. |
| `ramses_pvar_idx` | absent (1) | `2` | Number of the tracer file, `ic_pvar_00002`. See 7.2. |
| `filename` | `ics_zoom_dm` | `ics_zoom_hydro` | |

The seeds, `levelmin`, `levelmax`, `zstart` and the cosmology are unchanged, so the halo is the same. The mass of a level-9 dark-matter particle is split between dark matter and gas: $8.59\times10^{6}$ (dark matter) $+\ 1.58\times10^{6}$ (gas per cell) $= 1.02\times10^{7}\,h^{-1}M_\odot$, the particle mass of Part 2.

**No point shift is needed.** hast `decontaminate` already brought `results_part_wide` back to the frame of the parent box (`point_shift` in Part 2). MUSIC computes a new shift from the centroid of this region, as in Part 2.

### 7.2. The zoom tracer and the `ic_pvar` files

In a dark-matter zoom, RAMSES keeps refinement inside the high-resolution region with `mass_cut_refine`: only light particles trigger it. Gas cells have no particle mass, so a hydro zoom needs another way to tell the high-resolution region from the rest of the box. MUSIC provides it as a **passive scalar**, a tracer equal to `ramses_pvar_val` (default 1) in the refined region and 0 outside. The gas carries it along, so it follows the gas of the zoom region during the run.

**How RAMSES reads it.** RAMSES reads `ic_pvar_NNNNN` into hydro variable `nhydro + NNNNN`, where `nhydro = 5` in 3D (density, three momenta, energy). A missing file is skipped. In `hydro/init_flow_fine.f90`:

```fortran
if(ivar>nhydro.and.ivar<=nvar)then
   call title(ivar-nhydro,ncharvar)
   filename=TRIM(initfile(ilevel))//'/ic_pvar_'//TRIM(ncharvar)
endif
INQUIRE(file=filename,exist=ok_file3)
```

So the rule is: **file number = `ivar_refine` − 5**. MUSIC writes `ic_pvar_<ramses_pvar_idx>` and, in its suggested namelist, `ivar_refine = 5 + ramses_pvar_idx` and `var_cut_refine = 0.01 × ramses_pvar_val` (`src/plugins/output_grafic2.cc`).

**The trap: the metallicity slot.** Public RAMSES orders the variables after the five hydro ones as: non-thermal energies (`NENER`, 0 here), then **metallicity**, then the other passive scalars. The metallicity is `imetal = inener + nener = 6` (`hydro/hydro_parameters.f90`). With the MUSIC default (`ramses_pvar_idx = 1`), the tracer is written to `ic_pvar_00001` and read into variable 6: RAMSES would start with a metallicity of 1 (in units of the metal mass fraction) in the whole zoom region, and refine on the metallicity.

**The fix used here.**

| | Variable | Content | IC file |
|---|---|---|---|
| `imetal` | 6 | Metallicity | `ic_pvar_00001`: not written, so it starts at 0 |
| `ivar_refine` | 7 | Zoom tracer | `ic_pvar_00002`, from `ramses_pvar_idx = 2` |

RAMSES must then be compiled with one metal variable and one passive scalar, `NMETALS=1 NPSCAL=1`, i.e. `nvar = 5 + 1 + 1 = 7` (section 8.1). The same rule applies to any other build: choose `ivar_refine` as a variable that no physics module uses, and set `ramses_pvar_idx = ivar_refine − 5`.

**What the tracer does in RAMSES.** With `ivar_refine > 0` and `var_cut_refine = 0.01`, a cell is "in the zoom" if its tracer concentration (variable 7 divided by the density) is above 0.01:

- the initial mesh is refined in these cells (`amr/flag_utils.f90`, instead of `ic_refmap` used with `ivar_refine = 0` in Part 2);
- only the gas of these cells counts in the quasi-Lagrangian refinement, `m_refine` (`pm/rho_fine.f90`);
- stars form only in these cells (`pm/star_formation.f90`).

Dark matter is still limited by `mass_cut_refine`, as in Part 2.

### 7.3. Running MUSIC

```bash
cd MyFirstZoom/ZoomHydro/ICs
$MUSICPATH/MUSIC music.conf
```

It takes a few seconds. Check the log as in Part 2, and the tracer:

```bash
grep -E "are refined|shifted by|particle mass|mass/cell" music.conf_log.txt
ls ics_zoom_hydro/level_009/ic_pvar_*
grep -E "ivar_refine|var_cut_refine" ics_zoom_hydro/ramses.nml
```

1. **`… are refined`** is non-zero: 0.0398 on level 8, 0.00189 on level 7.
2. **`Domain will be shifted by (−37, 21, 24)`**, the same as in Part 2. The shift is recomputed from the centroid of each new region, so it can change by one cell from one zoom to the next. It does not here, so the dark-matter-only and hydro runs share the same frame and can be compared directly. **Any later step on this run (e.g. another decontamination) must take `point_shift` from this log.**
3. The tracer is `ic_pvar_00002`, and MUSIC suggests `ivar_refine=7` and `var_cut_refine=0.01`.

MUSIC does not write `ic_tempb`. RAMSES then starts the gas at the temperature of its cooling model at $z_\mathrm{start}$ (`Starting with T/mu (K) = 135.5` in the RAMSES log).

## 8. Hydrodynamical zoom run

### 8.1. Building RAMSES with metals and a passive scalar

Build a second executable next to the one of Parts 1–2. Building in a copy of `bin/` keeps the dark-matter objects untouched:

```bash
cd $RAMSESPATH
mkdir bin_hydro
cp bin/Makefile bin/clusters.mk bin_hydro/
cd bin_hydro
make NDIM=3 MPI=1 NMETALS=1 NPSCAL=1 EXEC=ramses_hydro
```

This gives `ramses_hydro3d`. At start-up it must print `Using solver = hydro with nvar =  7` and `imetal = 6`.

### 8.2. RAMSES namelist

The file is `MyFirstZoom/ZoomHydro/Run/ramses.nml`. It is the namelist of Part 2 with these changes:

| Parameter | Part 2 | Part 3 | Why |
|---|---|---|---|
| `hydro` | `.false.` | `.true.` | Gas. |
| `initfile(1:3)` | `../ICs/ics_zoom_dm/…` | `../ICs/ics_zoom_hydro/…` | |
| `aend` | 1.0 | 0.3333333 | Stop at $z = 2$ on a laptop. |
| `ivar_refine`, `var_cut_refine` | 0, – | 7, 0.01 | The zoom tracer of 7.2, as suggested by MUSIC. |
| `&HYDRO_PARAMS` | – | `muscl`, `courant_factor=0.8`, `slope_type=1`, `gamma=5/3`, `pressure_fix` | From `namelist/cosmo_gal.nml`. |
| `&COOLING_PARAMS` | – | `cooling=.true.`, `metal=.true.` | Idem. Metal-dependent cooling. |
| `&SF_PARAMS` | – | `eps_star=0.0204`, `n_star=0.1`, `T2_star=1d4` | Idem, see below. |
| `&FEEDBACK_PARAMS` | – | `eta_sn=0.1`, `yield=0.1` | Idem. Thermal supernova feedback, 10% of the stellar mass; 10% of the ejecta are metals. |

The physics follows the cosmological example of public RAMSES, `namelist/cosmo_gal.nml`, with one change. That file sets the star formation time scale `t_star = 8` (Gyr), which the current code no longer accepts (`Cannot match namelist object name t_star`). The current parameter is the efficiency per free-fall time, `eps_star`. The conversion is given in `hydro/read_hydro_params.f90`:

```fortran
! t_star=0.1635449d0*(n_star/0.1d0)**(-0.5d0)/eps_star
```

so `t_star = 8` with `n_star = 0.1` is `eps_star = 0.1635449/8 = 0.0204`. Star formation is switched on by `eps_star > 0`.

The other parameters are unchanged: `levelmax = 18` (cells of $\sim 140$ comoving pc, 47 physical pc at $z=2$), `m_refine`, `mass_cut_refine`, `nexpand`, the clump finder. As in `cosmo_gal.nml`, there is no Jeans refinement: the gas refines on mass only.

### 8.3. Running RAMSES

```bash
cd MyFirstZoom/ZoomHydro/Run
mpirun -np 16 $RAMSESPATH/bin_hydro/ramses_hydro3d ramses.nml > run.log
```

**Check the start of the log:**

```bash
grep -E "nvar|imetal|ic_pvar|Starting with" run.log
grep -A10 "Initial mesh structure" run.log | grep -E "Level +(7|8|9) "
```

```
 Using solver = hydro with nvar =  7
    imetal   =            6
 Reading file ../ICs/ics_zoom_hydro/level_009/ic_pvar_00002
 Starting with T/mu (K) =    135.51459709609756
```

Lines `File …/ic_tempb not found` are expected (7.3). Levels 8 and 9 must exist from the start, as in Part 2.

### 8.4. Results

*To be completed when the run has finished.*

---
**Production settings.** Same as in Part 2 (`levelmax = 11` in MUSIC, five `initfile` levels, `levelmax = 20`, `nexpand = 4,4,4,1,1`, `mass_cut_refine = 2.32831e-10`, `m_refine = 13*8.`, `nsubcycle = 1,1,1,1,1,1,2`), with `aend = 1`. The tracer setup does not depend on the resolution: `ramses_pvar_idx = 2`, `ivar_refine = 7`, `var_cut_refine = 0.01`, and RAMSES built with `NMETALS=1 NPSCAL=1`.